Ready for the tennis dataset
Update tennis dataset through tennis_dataset_update.py file

In [329]:
import pandas as pd
import matplotlib.pyplot as plt

tennis_df = pd.read_csv("/Users/chiangethan/.cache/kagglehub/datasets/dissfya/atp-tennis-2000-2023daily-pull/versions/911/atp_tennis.csv")
player_rank = pd.read_csv("dataset/tennisATPRanking.csv")
print(tennis_df.columns)
print(player_rank.columns)

Index(['Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of',
       'Player_1', 'Player_2', 'Winner', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2',
       'Odd_1', 'Odd_2', 'Score'],
      dtype='object')
Index(['Unnamed: 0', 'Name', 'Rank', 'Points', 'Country', 'Tournaments_Played',
       'Best_Rank'],
      dtype='object')


Data Wrangling For Each Dataset

In [330]:
#Manipulating tennis_df
tennis_df["Date"] = pd.to_datetime(tennis_df["Date"])

#Manipulatin player_rank
player_rank = player_rank.iloc[:, 1:]

def formatting(name):
    first_last = name.split(" ")
    if len(first_last) > 2:
        return first_last[2] + " " + first_last[0][0] + "."
    return first_last[1] + " " + first_last[0][0] + "."
player_rank["Name"] = player_rank["Name"].apply(formatting)

Framing the Data Base for Analysis Based on the Rank at 06/26/2025

The new dataframe match_with_top200 focuses on the usage of analyzing the consistency of each top 200 ATP ranking players

In [331]:
top_200_list= player_rank[player_rank["Rank"] <= 200]["Name"].to_list()
match_with_top200 = tennis_df[(tennis_df["Player_1"].isin(top_200_list)) | (tennis_df["Player_2"].isin(top_200_list))]

#Deleting the rows with duplicated player name that is not listed in the current top ATP 200 ranking
drop_old_Ruud_row = match_with_top200[((match_with_top200["Player_1"] == "Ruud C.") | (match_with_top200["Player_2"] == "Ruud C.")) & (match_with_top200["Date"].dt.year < 2015)].index
match_with_top200 = match_with_top200.drop(drop_old_Ruud_row)

match_with_top200 = match_with_top200.reset_index()

Creating dataframe that record the career stats for each player in ATP top 200

In [332]:
top_200_player_stat_df = pd.DataFrame({"Player": top_200_list, "Win": [0] * len(top_200_list),
                                       "Loss": [0] * len(top_200_list), "Matches": [0] * len(top_200_list)})
for i in range(len(match_with_top200)):
    loser = ""
    winner = match_with_top200.iloc[i]["Winner"]
    if match_with_top200.iloc[i]["Player_1"] == winner:
        loser = match_with_top200.iloc[i]["Player_2"]
    else:
        loser = match_with_top200.iloc[i]["Player_1"]
    
    if winner in top_200_list:
        top_200_player_stat_df.loc[top_200_player_stat_df["Player"] == loser, "Loss"] += 1
        top_200_player_stat_df.loc[top_200_player_stat_df["Player"] == loser, "Matches"] += 1

    if loser in top_200_list:
        top_200_player_stat_df.loc[top_200_player_stat_df["Player"] == winner, "Win"] += 1
        top_200_player_stat_df.loc[top_200_player_stat_df["Player"] == winner, "Matches"] += 1

top_200_player_stat_df

,Player,Win,Loss,Matches
0,Sinner J.,202,61,263
1,Alcaraz C.,181,48,229
2,Zverev A.,239,108,347
3,Draper J.,77,33,110
4,Fritz T.,165,107,272
...,...,...,...,...
195,Hassan B.,0,1,1
196,Droguet T.,3,4,7
197,Rodionov J.,2,12,14
198,Clarke J.,0,5,5


In [333]:
top_200_player_stat_df["Win_percent"] = (top_200_player_stat_df["Win"] / top_200_player_stat_df["Matches"]).round(2)
top_200_player_stat_df["Win_Loss_Ratio"] = (top_200_player_stat_df["Win"] / top_200_player_stat_df["Loss"]).round(2)
top_200_player_stat_df["Rank"] = top_200_player_stat_df.index + 1

top_200_player_stat_df.sort_values(by = "Win_percent", ascending = False).head(20)


,Player,Win,Loss,Matches,Win_percent,Win_Loss_Ratio,Rank
122,Zhang Z.,1,0,1,1.00,inf,123
5,Djokovic N.,281,47,328,0.86,5.98,6
1,Alcaraz C.,181,48,229,0.79,3.77,2
0,Sinner J.,202,61,263,0.77,3.31,1
3,Draper J.,77,33,110,0.70,2.33,4
8,Medvedev D.,214,93,307,0.70,2.30,9
2,Zverev A.,239,108,347,0.69,2.21,3
16,Mensik J.,45,21,66,0.68,2.14,17
135,Gaubas V.,2,1,3,0.67,2.00,136
14,Ruud C.,157,91,248,0.63,1.73,15


Creating consistency matrix to examine the consistency of players monthly, quaterly and yearly (mainly quaterly)

In [334]:
#Making sure each series has the similar round system
#so we can assign the uniform point for the winner of the round
series_type = match_with_top200["Series"].unique()
for series in series_type:
    types = match_with_top200[match_with_top200["Series"] == series]["Round"].unique().tolist()
    print(f"{series}: include {types}")

Masters: include ['1st Round', '2nd Round', '3rd Round', '4th Round', 'Quarterfinals', 'Semifinals', 'The Final']
Grand Slam: include ['1st Round', '2nd Round', '3rd Round', '4th Round', 'Quarterfinals', 'Semifinals', 'The Final']
International: include ['1st Round', '2nd Round', 'Quarterfinals', 'Semifinals', 'The Final', '3rd Round', 'Round Robin']
International Gold: include ['1st Round', '2nd Round', '3rd Round', 'Quarterfinals', 'Semifinals', 'The Final']
Masters Cup: include ['Round Robin', 'Semifinals', 'The Final']
ATP250: include ['1st Round', '2nd Round', 'Quarterfinals', 'Semifinals', 'The Final', '3rd Round']
ATP500: include ['1st Round', '2nd Round', 'Quarterfinals', 'Semifinals', 'The Final', '3rd Round']
Masters 1000: include ['1st Round', '2nd Round', '3rd Round', '4th Round', 'Quarterfinals', 'Semifinals', 'The Final']


In [335]:
#In 2007 there was a experiment of adding Robin Round into International series, which makes International Series appeared
#to have additional Round Robin
#After that Round Robin has never been used for International Series
#So we would remove year 2007 out of the analyzed datset
match_with_top200 = match_with_top200[match_with_top200["Date"].dt.year != 2007]

In [336]:
point_assign_dict = {'1st Round': 1, '2nd Round': 2, '3rd Round': 3, '4th Round': 4, 'Round Robin': 3, 'Quarterfinals': 5, 'Semifinals': 6, 'The Final': 7}
def assign(round):
    return point_assign_dict[round]
match_with_top200["Win_point"] = match_with_top200["Round"].apply(assign)

In [337]:
#Create the dataframe that collect player's individual record in match per row format
#So each match has one row for each player
individual_record_df = pd.DataFrame({"Date": [], "Tournament": [], "Series": [], "Round": [], "Player": [], 
                                     "Status": [], "Win_point": [], "Court": [], "Surface": [], "Score": []})
for i in range(len(match_with_top200)):
    curr = match_with_top200.iloc[i]
    winner = curr["Winner"]
    if curr["Player_1"] == winner:
        if curr["Player_1"] in top_200_list:
            individual_record_df.loc[(len(individual_record_df))] = [curr["Date"], curr["Tournament"], curr["Series"], curr["Round"], curr["Player_1"],
                                                                 "win", curr["Win_point"], curr["Court"], curr["Surface"], curr["Score"]]
        if curr["Player_2"] in top_200_list:
            individual_record_df.loc[(len(individual_record_df))] = [curr["Date"], curr["Tournament"], curr["Series"], curr["Round"], curr["Player_2"],
                                                                 "loss", 0, curr["Court"], curr["Surface"], curr["Score"]]
    else:
        if curr["Player_2"] in top_200_list:
            individual_record_df.loc[(len(individual_record_df))] = [curr["Date"], curr["Tournament"], curr["Series"], curr["Round"], curr["Player_2"],
                                                                 "win", curr["Win_point"], curr["Court"], curr["Surface"], curr["Score"]]
        if curr["Player_1"] in top_200_list:
            individual_record_df.loc[(len(individual_record_df))] = [curr["Date"], curr["Tournament"], curr["Series"], curr["Round"], curr["Player_1"],
                                                                 "loss", 0, curr["Court"], curr["Surface"], curr["Score"]]


In [338]:
# individual_record_df.groupby([individual_record_df["Date"].dt.month, "Player"])["Win_point"].agg(["sum", "mean", "std"]).head(50)
grand_performance_year = individual_record_df[(individual_record_df["Series"] == "Grand Slam") & 
                     (individual_record_df["Tournament"] == "French Open")].groupby(
                         [individual_record_df["Date"].dt.year, "Player"])["Win_point"].agg(["sum"])
player_consistency_french = grand_performance_year.groupby("Player")["sum"].agg(["sum", "std"]).sort_values("sum", ascending = False).head(50)
player_consistency_french["Play years"] = grand_performance_year.reset_index()["Player"].value_counts()
player_consistency_french["Point per year"] = (player_consistency_french["sum"] / player_consistency_french["Play years"]).round(2)
player_consistency_french

,sum,std,Play years,Point per year
Player,,,,
Djokovic N.,294,7.733046,20,14.70
Wawrinka S.,116,7.773850,19,6.11
Zverev A.,101,6.471304,10,10.10
Alcaraz C.,78,9.813256,5,15.60
Monfils G.,77,4.445751,17,4.53
Tsitsipas S.,67,6.710274,9,7.44
Ruud C.,63,8.576338,8,7.88
Sinner J.,59,7.194906,6,9.83
Gasquet R.,52,2.817240,20,2.60


In [326]:
Gslam_df = individual_record_df[individual_record_df["Series"] == "Grand Slam"]
date_player_tour = Gslam_df.groupby([Gslam_df["Date"].dt.year, 
                                               "Player", "Tournament"])["Win_point"].agg(["sum"])
Gslam_consistency = date_player_tour.groupby("Player")["sum"].agg(["sum", "std"])
Gslam_consistency["times_of_Gslam"] = date_player_tour.reset_index()["Player"].value_counts()
Gslam_consistency["points_per_time"] = (Gslam_consistency["sum"] / Gslam_consistency["times_of_Gslam"]).round(2)
Gslam_consistency.sort_values("sum", ascending=False)

,sum,std,times_of_Gslam,points_per_time
Player,,,,
Djokovic N.,1240,9.335539,76,16.32
Wawrinka S.,355,6.655991,69,5.14
Zverev A.,270,6.006070,39,6.92
Sinner J.,262,9.513910,24,10.92
Alcaraz C.,259,10.599846,19,13.63
...,...,...,...,...
O'Connell C.,0,0.000000,2,0.00
McCabe J.,0,NaN,1,0.00
McDonald M.,0,0.000000,3,0.00
